# A/B Testing and Statistical Analysis: Conversion Optimization

This notebook implements a complete A/B test on synthetic data for a landing page conversion experiment. It includes hypothesis setup, frequentist analysis, Bayesian comparison, and power analysis.

## 1. Imports & Synthetic Data Generation

We'll generate 10,000 user interactions across two groups (Control and Variant).

In [ ]:

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import TTestIndPower

# Set seed for reproducibility
np.random.seed(42)

# Parameters
n_users = 10000
p_a = 0.12  # Control conversion rate (12%)
p_b = 0.14  # Variant conversion rate (14%)

# Generate conversion data
group_a = np.random.binomial(1, p_a, n_users // 2)
group_b = np.random.binomial(1, p_b, n_users // 2)

# Create DataFrame
df = pd.DataFrame({
    'group': ['A'] * (n_users // 2) + ['B'] * (n_users // 2),
    'converted': np.concatenate([group_a, group_b])
})

print(df.groupby('group')['converted'].agg(['count', 'mean', 'sum']))


## 2. Hypothesis Setup

**Null Hypothesis (H₀):** There is no difference in the conversion rate between Group A and Group B.

**Alternative Hypothesis (H₁):** Group B has a higher conversion rate than Group A.

## 3. Frequentist Analysis: Chi-Square and T-test

Since our outcome is binary, we use the Chi-square test for independence or a Z-test for proportions.

In [ ]:

# Chi-Square Test
contingency_table = pd.crosstab(df['group'], df['converted'])
chi2, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

# T-test (often used for proportions as well with large samples)
t_stat, p_val_t = stats.ttest_ind(group_a, group_b)

print(f"Chi-square p-value: {p_val_chi2:.4f}")
print(f"T-test p-value: {p_val_t:.4f}")


## 4. Confidence Intervals & Effect Size (Cohen's d)

We calculate the 95% confidence intervals and Cohen's d to understand the magnitude of the effect.

In [ ]:

# 95% Confidence Intervals for proportions
def proportion_ci(p, n, z=1.96):
    se = np.sqrt(p * (1 - p) / n)
    return (p - z * se, p + z * se)

ci_a = proportion_ci(group_a.mean(), len(group_a))
ci_b = proportion_ci(group_b.mean(), len(group_b))

print(f"Group A 95% CI: {ci_a}")
print(f"Group B 95% CI: {ci_b}")

# Cohen's d
def cohens_d(x, y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)

d = cohens_d(group_b, group_a)
print(f"Cohen's d: {d:.4f}")


## 5. Statistical Power Analysis

Was our sample size sufficient to detect this change at an 80% power level?

In [ ]:

analysis = TTestIndPower()
power = analysis.power(effect_size=d, nobs1=n_users//2, ratio=1.0, alpha=0.05)
print(f"Statistical Power: {power:.4f}")

# Plotting sample size vs power
analysis.plot_power(dep_var='nobs', nobs=np.arange(100, 10000, 100), effect_size=[0.05, 0.1, 0.2])
plt.title('Power vs Sample Size')
plt.show()


## 6. Bayesian Comparison

Using a Beta-Binomial conjugate prior to estimate the probability that B > A.

In [ ]:

from scipy.stats import beta

# Prior: Beta(1, 1) - Uniform prior
alpha_prior, beta_prior = 1, 1

# Posterior: Beta(alpha_prior + successes, beta_prior + failures)
a_successes, a_failures = group_a.sum(), len(group_a) - group_a.sum()
b_successes, b_failures = group_b.sum(), len(group_b) - group_b.sum()

posterior_a = beta(alpha_prior + a_successes, beta_prior + a_failures)
posterior_b = beta(alpha_prior + b_successes, beta_prior + b_failures)

# Sampling from posteriors
samples = 100000
samples_a = posterior_a.rvs(samples)
samples_b = posterior_b.rvs(samples)

prob_b_better_a = (samples_b > samples_a).mean()
print(f"Probability that Variant is better than Control: {prob_b_better_a:.4f}")

# Plotting Posteriors
plt.figure(figsize=(10, 6))
sns.kdeplot(samples_a, label='Posterior A', fill=True)
sns.kdeplot(samples_b, label='Posterior B', fill=True)
plt.title('Bayesian Posteriors for Conversion Rates')
plt.legend()
plt.show()
